# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print("Title:", metadata.name)
print("Description:", metadata.description)
print("Published date:", metadata.datePublished)
print("Keywords:", metadata.keywords)


## 2. Data Overview
Review available record sets, their fields, and corresponding `@id` values.

In [ ]:
# Check all record sets defined in the dataset metadata, referencing by '@id'
# If the dataset has record sets, they will be accessible via metadata.recordSet
record_sets = []
if getattr(metadata, 'recordSet', None):
    for rs in metadata.recordSet:
        print(f"RecordSet @id: {rs['@id']}")
        print(f" RecordSet name: {rs.get('name', 'N/A')}")
        fields = rs.get('field', [])
        print(f" Fields (@id): {[f['@id'] for f in fields]}")
        record_sets.append(rs['@id'])
else:
    print("No record sets found in the metadata.")

# If record_sets is empty, manually list a plausible record set for demonstration:
if not record_sets:
    # Using plausible value based on dataset structure
    record_sets = ['https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json#MentalHealthSurvey']
    print(f"Using example RecordSet: {record_sets[0]}")

# Preview a few records (by @id)
for rec in dataset.records(record_set=record_sets[0]):
    print(rec)
    break


## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. All operations reference entities by their `@id`.

In [ ]:
# Extract data from the main record set
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for RecordSet {record_set_id}:")
        print(df.columns.tolist())
        print(df.head())
    else:
        print(f"No records found for RecordSet {record_set_id}.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filter records based on a numeric mental health score field
- Normalize numeric values
- Group data by demographic attributes (by `@id`)

All fields are referenced via their `@id`.

In [ ]:
# Example field @ids (assumed based on typical mental health survey columns)
# Replace these with actual @ids from metadata overview

record_set_id = record_sets[0]
df = dataframes.get(record_set_id)

# Example field @id for GAD-7 total score, replace if different in actual schema
numeric_field_id = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json#gad7_total_score'
# Example group-by field @id for gender
group_field_id = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json#gender'

# Use the actual column names if available
if df is not None:
    # Check if field exists as a column
    if numeric_field_id in df.columns:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        col_norm = f"{numeric_field_id}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, col_norm]].head())

        # Group by a demographic attribute
        if group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print(f"Field {numeric_field_id} not found in data columns.")
else:
    print(f"No dataframe found for RecordSet {record_set_id}.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Visualize the distribution of the GAD-7 scores (referenced by @id)
if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    df[numeric_field_id].hist(bins=20, color='skyblue')
    plt.title('Distribution of GAD-7 Total Scores')
    plt.xlabel('GAD-7 Total Score (@id)')
    plt.ylabel('Frequency')
    plt.show()
else:
    print("Unable to plot: DataFrame or field not found.")


## 6. Conclusion
This notebook demonstrates loading, overview, and basic exploration of the FAIR² Mental Health Survey Dataset using `mlcroissant`. Key fields and groupings are referenced strictly by their `@id`, ensuring reproducibility and clarity. For further analysis, consult the full schema for exact column identities and types.

- We loaded the metadata and records using the schema URL.
- Explored available record sets and fields, referencing each by `@id`.
- Performed basic filtering, normalization, and grouping on a sample numeric field.
- Visualized the distribution of mental health scores.

For more advanced analyses, refer directly to the Croissant schema and use field @ids for all manipulations.